# CS229 L10 — EM Derivation + Principal Component Analysis

**Video:** Spring 2026 · [YouTube](https://www.youtube.com/watch?v=sUS-eTa0l6s)  
**Instructor:** Tengyu Ma  
**Topics:** EM derivation (Q(z) trick, ELBO, tightness), EM convergence, PCA (eigendecomposition, projection, explained variance)

---

| Section | Content |
|---|---|
| 1 | EM Motivation — why MLE over latent variables is hard |
| 2 | Q(z) Trick + ELBO via Jensen's Inequality |
| 3 | Tightness + E-step as Posterior |
| 4 | EM Convergence — monotone likelihood proof |
| 5 | Full GMM-EM verified against sklearn |
| 6 | PCA — Motivation, Centering, Rescaling |
| 7 | PCA — Eigendecomposition of Covariance |
| 8 | PCA — Projection, Reconstruction, Explained Variance |
| 9 | PCA Pitfalls — eigenvalue gap, missing centering, scale sensitivity |

## 1. EM Motivation — Why Latent Variables Make MLE Hard

In supervised learning we maximized $\ell(\theta) = \sum_i \log p(x^{(i)}, y^{(i)}; \theta)$.  
The labels $y^{(i)}$ are **observed** — every term in the gradient is tractable.

In GMM, each point $x^{(i)}$ has a **latent** cluster assignment $z^{(i)} \in \{1,\ldots,k\}$ that we never observe.  
The log-likelihood marginalizes over $z$:

$$\ell(\theta) = \sum_{i=1}^n \log p(x^{(i)};\theta) = \sum_{i=1}^n \log \sum_{z=1}^k p(x^{(i)}, z;\theta)$$

The **log-of-sum** has no closed-form gradient — we cannot directly differentiate and set to zero.

### Factorization assumption

EM requires knowing the factorization structure:
$$p(x;\theta) = \sum_z p(x,z;\theta) = \sum_z p(z;\theta)\,p(x|z;\theta)$$

For GMM: $p(z=j;\theta) = \phi_j$ and $p(x|z=j;\theta) = \mathcal{N}(x;\mu_j, \Sigma_j)$.  
This is a modeling assumption — we hypothesize that latent cluster structure exists with a known shape.

## 2. Q(z) Trick + ELBO via Jensen's Inequality

**Key idea:** introduce any distribution $Q_i(z)$ over the latent variable $z$, with $Q_i(z) \geq 0$, $\sum_z Q_i(z) = 1$.  
Use it to rewrite the log-likelihood as an expectation, then apply Jensen.

$$\log p(x^{(i)};\theta) = \log \sum_z p(x^{(i)},z;\theta)
= \log \sum_z Q_i(z) \cdot \frac{p(x^{(i)},z;\theta)}{Q_i(z)}$$

The right-hand side is $\log\, \mathbb{E}_{z \sim Q_i}\!\left[\frac{p(x^{(i)},z;\theta)}{Q_i(z)}\right]$.

**Jensen's inequality** for a concave function $f$ (here $f = \log$):
$$f\bigl(\mathbb{E}[X]\bigr) \geq \mathbb{E}[f(X)]$$

Applying Jensen:
$$\log p(x^{(i)};\theta)
\geq \sum_z Q_i(z) \log\frac{p(x^{(i)},z;\theta)}{Q_i(z)}
\;=:\; \mathcal{L}_i(Q_i, \theta)$$

Summing over all data points defines the **ELBO** (Evidence Lower BOund):

$$\mathcal{L}(Q,\theta) = \sum_{i=1}^n \sum_z Q_i(z) \log\frac{p(x^{(i)},z;\theta)}{Q_i(z)} \leq \ell(\theta)$$

This lower bound holds for **any** valid $Q_i$. The bound is the backbone of VAEs, diffusion models, and all variational inference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Visualize Jensen's inequality for concave log ---
x = np.linspace(0.1, 3.0, 300)
f = np.log(x)                        # concave function

# Two support points for Q
x1, x2 = 0.5, 2.5
q1, q2 = 0.4, 0.6
Ex = q1 * x1 + q2 * x2              # E[X]
Efx = q1 * np.log(x1) + q2 * np.log(x2)  # E[f(X)]  — lower
fEx = np.log(Ex)                     # f(E[X])  — upper

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, f, 'steelblue', lw=2, label=r'$f(x) = \log x$')

# Chord connecting (x1,f(x1)) to (x2,f(x2))
ax.plot([x1, x2], [np.log(x1), np.log(x2)], 'k--', lw=1, label='chord (chord ≤ curve)')

# Jensen points
ax.scatter([Ex], [fEx], color='red',    zorder=5, s=70, label=rf'$f(\mathbb{{E}}[X])={fEx:.3f}$')
ax.scatter([Ex], [Efx], color='orange', zorder=5, s=70, label=rf'$\mathbb{{E}}[f(X)]={Efx:.3f}$ (ELBO)')
ax.annotate('', xy=(Ex, fEx), xytext=(Ex, Efx),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=1.5))
ax.text(Ex + 0.08, (fEx + Efx) / 2, 'gap\n(Jensen slack)', fontsize=9, color='purple')

ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title("Jensen's Inequality: $f(\\mathbb{E}[X]) \\geq \\mathbb{E}[f(X)]$ for concave $f$")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'f(E[X]) = {fEx:.4f}   ≥   E[f(X)] = {Efx:.4f}   (gap = {fEx - Efx:.4f})')

## 3. Tightness — E-Step as Posterior

Jensen's inequality is **tight** (equality holds) if and only if the argument of $f$ is constant w.r.t. $z$:

$$\frac{p(x^{(i)},z;\theta)}{Q_i(z)} = c_i \quad \text{(constant in } z \text{)}$$

This means $Q_i(z) \propto p(x^{(i)},z;\theta)$.  
Since $Q_i$ must sum to 1, the unique tight choice is:

$$\boxed{Q_i(z) = p\!\left(z \mid x^{(i)}; \theta\right)}$$

**The E-step is exactly computing the posterior over $z$.**  When $Q_i$ equals the current posterior, the ELBO touches $\ell(\theta^{(t)})$ from below — the lower bound is tight at the current parameters.

### The EM Loop

| Step | Operation | Interpretation |
|---|---|---|
| **E-step** | $Q_i^{(t)}(z) = p(z\mid x^{(i)};\theta^{(t)})$ | Soft cluster assignments using current $\theta$ |
| **M-step** | $\theta^{(t+1)} = \arg\max_\theta \mathcal{L}(Q^{(t)}, \theta)$ | Weighted MLE with frozen $Q$ |

After the E-step the ELBO is tight at $\theta^{(t)}$.  The M-step can only increase $\mathcal{L}$, which cannot exceed $\ell(\theta)$, so the likelihood is non-decreasing.

## 4. EM Convergence — Monotone Likelihood Proof

**Claim:** $\ell(\theta^{(t+1)}) \geq \ell(\theta^{(t)})$ at every iteration.

**Proof:**

1. After E-step at $\theta^{(t)}$: $Q_i^{(t)} = p(z|x^{(i)};\theta^{(t)})$ → bound is tight:
$$\ell(\theta^{(t)}) = \mathcal{L}(Q^{(t)}, \theta^{(t)})$$

2. M-step maximizes $\mathcal{L}$ over $\theta$:
$$\mathcal{L}(Q^{(t)}, \theta^{(t+1)}) \geq \mathcal{L}(Q^{(t)}, \theta^{(t)}) = \ell(\theta^{(t)})$$

3. ELBO lower-bounds true likelihood for any $Q$:
$$\ell(\theta^{(t+1)}) \geq \mathcal{L}(Q^{(t)}, \theta^{(t+1)})$$

4. Chaining: $\ell(\theta^{(t+1)}) \geq \ell(\theta^{(t)})$. $\square$

**Important caveats:**
- EM converges to a **local maximum**, not necessarily global.
- Multiple random restarts help escape bad local optima.
- Convergence criterion: $|\ell(\theta^{(t+1)}) - \ell(\theta^{(t)})| < \epsilon$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Visualize EM monotone likelihood increase ---
# Synthetic 1D GMM: two well-separated Gaussians
np.random.seed(0)
n = 200
z_true = np.random.binomial(1, 0.4, n)
x = np.where(z_true == 0, np.random.normal(-2, 0.8, n), np.random.normal(2, 0.8, n))

def log_likelihood_1d(x, phi, mu1, mu2, sig):
    """GMM log-likelihood for 2-component 1D mixture."""
    p1 = phi       * (1/(np.sqrt(2*np.pi)*sig)) * np.exp(-0.5*((x-mu1)/sig)**2)
    p2 = (1-phi)   * (1/(np.sqrt(2*np.pi)*sig)) * np.exp(-0.5*((x-mu2)/sig)**2)
    return np.sum(np.log(p1 + p2 + 1e-300))

# EM loop tracking log-likelihood
phi, mu1, mu2, sig = 0.5, -0.5, 0.5, 1.0   # intentionally bad init
ll_history = [log_likelihood_1d(x, phi, mu1, mu2, sig)]

for _ in range(40):
    # E-step: soft assignments
    p1 = phi     * np.exp(-0.5*((x-mu1)/sig)**2)
    p2 = (1-phi) * np.exp(-0.5*((x-mu2)/sig)**2)
    w  = p1 / (p1 + p2 + 1e-300)       # w[i] = P(z_i=0 | x_i)

    # M-step: weighted MLE
    phi = w.mean()
    mu1 = (w * x).sum() / (w.sum() + 1e-300)
    mu2 = ((1-w) * x).sum() / ((1-w).sum() + 1e-300)
    sig = np.sqrt((w*(x-mu1)**2 + (1-w)*(x-mu2)**2).sum() / n)

    ll_history.append(log_likelihood_1d(x, phi, mu1, mu2, sig))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ll_history, 'o-', color='steelblue', ms=4, lw=1.5)
ax.set_xlabel('EM iteration'); ax.set_ylabel('Log-likelihood')
ax.set_title('EM: Monotone Log-Likelihood Increase (1D GMM)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Verify strictly non-decreasing
diffs = np.diff(ll_history)
print(f'All increments ≥ 0: {np.all(diffs >= -1e-8)}')
print(f'Converged μ₁={mu1:.3f}, μ₂={mu2:.3f}, φ={phi:.3f}, σ={sig:.3f}')

## 5. Full GMM-EM from Scratch — 2D, Verified Against sklearn

Full multivariate GMM-EM with diagonal $\Sigma_j$ (for numerical stability), compared against sklearn's `GaussianMixture`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.mixture import GaussianMixture

# --- Generate 2D three-cluster data ---
np.random.seed(7)
centers = np.array([[-3, -2], [0, 3], [3, -1]], dtype=float)
covs    = [np.array([[1.2, 0.4],[0.4, 0.6]]),
           np.array([[0.8,-0.3],[-0.3,1.0]]),
           np.array([[0.5, 0.0],[0.0, 1.5]])]
ns   = [120, 90, 150]
X_parts = [np.random.multivariate_normal(centers[j], covs[j], ns[j]) for j in range(3)]
X    = np.vstack(X_parts)
K, N, D = 3, X.shape[0], X.shape[1]

def gaussian_pdf(X, mu, Sigma):
    """Multivariate Gaussian density for each row of X."""
    d = X - mu                              # (N, D)
    Sinv = np.linalg.inv(Sigma)
    det  = np.linalg.det(Sigma)
    maha = np.einsum('ni,ij,nj->n', d, Sinv, d)  # Mahalanobis^2
    return np.exp(-0.5 * maha) / (np.sqrt((2*np.pi)**D * det) + 1e-300)

def plot_ellipse(ax, mu, Sigma, color, n_std=2):
    vals, vecs = np.linalg.eigh(Sigma)
    angle = np.degrees(np.arctan2(*vecs[:, -1][::-1]))
    w, h  = 2 * n_std * np.sqrt(vals)
    ell   = Ellipse(mu, w, h, angle=angle, color=color, fill=False, lw=2)
    ax.add_patch(ell)

# --- GMM-EM from scratch ---
# Init with K-Means centroids
from sklearn.cluster import KMeans
km   = KMeans(K, n_init=5, random_state=0).fit(X)
phi  = np.ones(K) / K
mu   = km.cluster_centers_.copy()
Sig  = np.array([np.eye(D) for _ in range(K)])

ll_hist = []
for it in range(60):
    # E-step
    W = np.column_stack([phi[j] * gaussian_pdf(X, mu[j], Sig[j]) for j in range(K)])
    ll_hist.append(np.sum(np.log(W.sum(axis=1) + 1e-300)))
    W /= W.sum(axis=1, keepdims=True) + 1e-300   # normalize → soft assignments

    # M-step
    Nj = W.sum(axis=0) + 1e-6                     # effective cluster counts
    phi = Nj / N
    for j in range(K):
        mu[j]  = (W[:, j:j+1] * X).sum(axis=0) / Nj[j]
        d      = X - mu[j]
        Sig[j] = (W[:, j:j+1] * d).T @ d / Nj[j] + 1e-4 * np.eye(D)

# Compare log-likelihoods with sklearn
skl = GaussianMixture(K, n_init=5, random_state=0).fit(X)
ll_ours = ll_hist[-1]
ll_skl  = skl.score(X) * N
print(f'Our EM   log-likelihood: {ll_ours:.2f}')
print(f'sklearn  log-likelihood: {ll_skl:.2f}')
print(f'Difference: {abs(ll_ours - ll_skl):.2f}  (small = matched)')

# --- Plot ---
colors = ['#e74c3c', '#2ecc71', '#3498db']
labels = W.argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for j in range(K):
    mask = labels == j
    ax.scatter(X[mask, 0], X[mask, 1], c=colors[j], s=10, alpha=0.6)
    plot_ellipse(ax, mu[j], Sig[j], colors[j])
    ax.scatter(*mu[j], marker='x', c='black', s=80, zorder=5)
ax.set_title('GMM-EM from Scratch (2D)')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.grid(alpha=0.2)

ax = axes[1]
ax.plot(ll_hist, 'steelblue', lw=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('Log-likelihood')
ax.set_title('EM Convergence Curve')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. PCA — Motivation, Centering, Rescaling

### Why PCA?

High-dimensional data often lies near a **low-dimensional subspace**. PCA finds that subspace.

Uses:
- **Dimensionality reduction:** $d = 10^4$ features → $k = 50$ principal components
- **Visualization:** project to 2D/3D for inspection
- **Compression / denoising:** keep top-$k$ directions, discard the rest

### Pre-processing: Centering (required)

PCA finds directions of **maximal variance**. Without centering, the first component points toward the data mean instead of the direction of spread.

$$\tilde{x}^{(i)} = x^{(i)} - \hat{\mu}, \qquad \hat{\mu} = \frac{1}{n}\sum_{i=1}^n x^{(i)}$$

### Pre-processing: Rescaling (often needed)

If features have different units (e.g., feet vs. miles, salary vs. age), one feature dominates the covariance.  
Divide each feature by its empirical standard deviation so all dimensions are on the same scale:

$$\hat{x}^{(i)}_j = \frac{\tilde{x}^{(i)}_j}{\hat{\sigma}_j}, \qquad \hat{\sigma}_j = \sqrt{\frac{1}{n}\sum_i (\tilde{x}^{(i)}_j)^2}$$

This is **not** always necessary — if features are naturally commensurate (e.g., pixel intensities), skip it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Demonstrate centering effect ---
np.random.seed(3)
n = 150
# Correlated data far from origin
t  = np.random.uniform(-2, 2, n)
X_raw = np.column_stack([5 + t + 0.2*np.random.randn(n),
                         8 + 0.5*t + 0.2*np.random.randn(n)])

mu   = X_raw.mean(axis=0)
X_c  = X_raw - mu                       # centered

def first_pc(X):
    """Return the first principal component (top eigenvector of X^T X)."""
    C  = X.T @ X / len(X)
    vals, vecs = np.linalg.eigh(C)
    return vecs[:, -1]                   # eigenvector of largest eigenvalue

u_raw = first_pc(X_raw)
u_cen = first_pc(X_c)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, X_plot, u, title, origin in [
    (axes[0], X_raw, u_raw, 'Without centering', X_raw.mean(axis=0)),
    (axes[1], X_c,   u_cen, 'With centering',    np.zeros(2))
]:
    ax.scatter(X_plot[:,0], X_plot[:,1], alpha=0.4, s=15, c='steelblue')
    scale = 2.0
    ax.annotate('', xy=origin + scale*u, xytext=origin - scale*u,
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_title(title); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

plt.suptitle('Centering Fixes PC Direction', fontsize=12)
plt.tight_layout()
plt.show()

print(f'PC without centering: {u_raw}')
print(f'PC with centering:    {u_cen}  ← points along data spread')

## 7. PCA — Eigendecomposition of Sample Covariance

### Sample covariance matrix

After centering, the sample covariance is:
$$\hat{\Sigma} = \frac{1}{n} X^T X \in \mathbb{R}^{d \times d}$$

$\hat{\Sigma}$ is symmetric positive semi-definite, so it has an eigendecomposition:
$$\hat{\Sigma} = U \Lambda U^T, \qquad \lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d \geq 0$$

$u_j$ = $j$-th eigenvector = $j$-th principal component direction.  
$\lambda_j$ = variance explained in direction $u_j$.

### Why eigenvectors maximize projected variance

We want $u$ that maximizes $\frac{1}{n}\sum_i (u^T x^{(i)})^2 = u^T \hat{\Sigma} u$ subject to $\|u\|=1$.

Expanding in the eigenbasis $u = \sum_j \alpha_j u_j$ with $\|u\|^2 = \sum_j \alpha_j^2 = 1$:
$$u^T \hat{\Sigma} u = \sum_j \lambda_j \alpha_j^2 \leq \lambda_1 \sum_j \alpha_j^2 = \lambda_1$$

Equality when $\alpha_1 = 1$, i.e., $u = u_1$. The **top eigenvector** is the first principal component.

Iterating (with the constraint that $u_2 \perp u_1$): the $k$-th PC is the $k$-th eigenvector.

### Total variance and trace

$$\text{Total variance} = \text{tr}(\hat{\Sigma}) = \sum_{j=1}^d \lambda_j$$

The trace can be computed as $\sum_j \hat{\Sigma}_{jj}$ in O(d) — no eigendecomposition needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Eigendecomposition of 2D covariance: geometric picture ---
np.random.seed(42)
n = 300
# Data with known covariance structure
Sigma_true = np.array([[3.0, 1.5], [1.5, 1.2]])
L = np.linalg.cholesky(Sigma_true)
X = (L @ np.random.randn(2, n)).T     # shape (n, 2), zero-mean

# Sample covariance
mu_hat  = X.mean(axis=0)
Xc      = X - mu_hat
Sigma_hat = Xc.T @ Xc / n

# Eigendecomposition
eigvals, eigvecs = np.linalg.eigh(Sigma_hat)   # ascending order
idx    = np.argsort(eigvals)[::-1]             # sort descending
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

print('Sample covariance:')
print(Sigma_hat.round(3))
print(f'\nEigenvalues: λ₁={eigvals[0]:.3f}, λ₂={eigvals[1]:.3f}')
print(f'Total variance (trace): {eigvals.sum():.3f}')
print(f'PC1 explains {eigvals[0]/eigvals.sum()*100:.1f}% of variance')
print(f'\nPC1 direction: {eigvecs[:,0].round(4)}')
print(f'PC2 direction: {eigvecs[:,1].round(4)}')

# --- Visualize ---
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(Xc[:,0], Xc[:,1], alpha=0.3, s=10, c='steelblue')

colors = ['red', 'orange']
for j in range(2):
    scale = np.sqrt(eigvals[j]) * 2
    v = eigvecs[:, j]
    ax.annotate('', xy=scale*v, xytext=-scale*v,
                arrowprops=dict(arrowstyle='->', color=colors[j], lw=2.5))
    ax.text(*(scale*v*1.1), f'PC{j+1}\n(λ={eigvals[j]:.2f})',
            fontsize=9, color=colors[j], ha='center')

ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.set_title('Principal Components = Eigenvectors of $\\hat{\\Sigma}$')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
plt.tight_layout()
plt.show()

## 8. PCA — Projection, Reconstruction, Explained Variance

### Projection to $k$ dimensions

Let $U_k = [u_1 | \cdots | u_k] \in \mathbb{R}^{d \times k}$ (columns = top-$k$ PCs).

$$z^{(i)} = U_k^T \tilde{x}^{(i)} \in \mathbb{R}^k \qquad \text{(low-dim code)}$$
$$\hat{x}^{(i)} = U_k z^{(i)} = U_k U_k^T \tilde{x}^{(i)} \in \mathbb{R}^d \qquad \text{(reconstruction)}$$

The reconstruction error equals the discarded variance:
$$\frac{1}{n}\sum_i \|\tilde{x}^{(i)} - \hat{x}^{(i)}\|^2 = \sum_{j=k+1}^d \lambda_j$$

### Explained variance ratio

$$\text{EVR}(k) = \frac{\sum_{j=1}^k \lambda_j}{\sum_{j=1}^d \lambda_j}$$

**Scree plot:** plot $\lambda_j$ vs. $j$ — look for an "elbow" where eigenvalues flatten.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# --- PCA on 8x8 digits (64-dim → visualize scree + reconstructions) ---
digits = load_digits()
X_raw  = digits.data.astype(float)      # (1797, 64)

# Centering
mu     = X_raw.mean(axis=0)
X      = X_raw - mu

# Full covariance eigendecomposition
Sigma   = X.T @ X / len(X)
eigvals, eigvecs = np.linalg.eigh(Sigma)
idx     = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]              # columns = PCs in descending order

evr     = eigvals / eigvals.sum()
cumevr  = evr.cumsum()

# How many components for 90%, 95%, 99%?
for threshold in [0.90, 0.95, 0.99]:
    k_needed = int(np.searchsorted(cumevr, threshold)) + 1
    print(f'{threshold*100:.0f}% variance explained → k = {k_needed} components (from 64)')

# --- Reconstructions at different k ---
sample_idx = 42
x_orig     = X[sample_idx]

fig, axes = plt.subplots(2, 5, figsize=(13, 5))

ks = [1, 2, 4, 8, 16, 24, 32, 40, 48, 64]
for ax, k in zip(axes.ravel(), ks):
    Uk    = eigvecs[:, :k]
    z     = Uk.T @ x_orig
    x_rec = Uk @ z + mu
    ax.imshow(x_rec.reshape(8, 8), cmap='gray_r', vmin=0, vmax=16)
    err   = np.linalg.norm(x_orig - (x_rec - mu))**2
    ax.set_title(f'k={k}\nEVR={cumevr[k-1]*100:.0f}%', fontsize=8)
    ax.axis('off')

plt.suptitle('Digit Reconstruction vs Number of PCs', fontsize=12)
plt.tight_layout()
plt.show()

# --- Scree plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, 33), evr[:32], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot (top 32 PCs)'); axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(range(1, 65), cumevr, 'o-', ms=3, lw=1.5, color='steelblue')
for t, c in [(0.90,'red'),(0.95,'orange'),(0.99,'green')]:
    axes[1].axhline(t, color=c, ls='--', lw=1, label=f'{t*100:.0f}%')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Cumulative EVR')
axes[1].set_title('Cumulative Explained Variance'); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. PCA Pitfalls — Eigenvalue Gap, Centering, Scale

### Pitfall 1: Eigenvalue gap instability

If $\lambda_j \approx \lambda_{j+1}$, there is no single principal component — any vector in the shared eigenspace is a maximizer.  
In practice this means: re-running PCA on slightly perturbed data gives a **different** PC direction.  
If downstream models use these coordinates as features, predictions become unreliable.

**Diagnosis:** Check the scree plot. If eigenvalues plateau early without a clear elbow, PCA coordinates are not meaningful.

### Pitfall 2: Forgetting to center

Without centering, $X^T X$ contains $n\hat{\mu}\hat{\mu}^T$ as a dominant rank-1 term.  
The first PC points toward the mean, not the direction of spread.

### Pitfall 3: Scale sensitivity

If feature $j$ has variance $\hat{\sigma}_j^2$ that is much larger than others, it dominates $\hat{\Sigma}$.  
Rescaling by $\hat{\sigma}_j$ before PCA makes eigenvalues comparable across dimensions.

### Pitfall 4: Dimensionality reduction ≠ always better

PCA assumes the discarded dimensions are noise. If the task-relevant signal lives in small-variance directions (e.g., rare-class discriminative features), PCA can hurt downstream performance.

### EM vs PCA — structural comparison

| Property | EM (GMM) | PCA |
|---|---|---|
| Substructure assumed | Latent cluster labels $z$ | Linear subspace |
| Objective | MLE via ELBO surrogate | Maximize projected variance |
| Convergence | Monotone (local max) | Closed-form eigendecomposition |
| When it goes wrong | Wrong K, bad init | Eigenvalue plateau, no centering |
| Modern connections | VAEs, diffusion, HMMs | SVD, randomized SVD, kernel PCA |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Demonstrate eigenvalue gap instability ---
np.random.seed(0)
n = 200

def pc1_angle(lam1, lam2, n_trials=30):
    """Run PCA n_trials times on noise-perturbed data; return std of PC1 angle."""
    angles = []
    for _ in range(n_trials):
        Sigma = np.diag([lam1, lam2])
        L = np.linalg.cholesky(Sigma)
        X = (L @ np.random.randn(2, n)).T
        S = X.T @ X / n
        vals, vecs = np.linalg.eigh(S)
        u1 = vecs[:, -1]
        angles.append(np.degrees(np.arctan2(u1[1], u1[0])) % 180)
    return np.std(angles)

# Vary λ₁ while keeping λ₁+λ₂=4
lam1_vals = np.linspace(2.01, 3.8, 20)
gaps      = lam1_vals - (4 - lam1_vals)   # λ₁ - λ₂
stds      = [pc1_angle(l1, 4-l1) for l1 in lam1_vals]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gaps, stds, 'o-', color='steelblue', ms=5)
axes[0].set_xlabel('Eigenvalue gap (λ₁ − λ₂)')
axes[0].set_ylabel('Std of PC1 angle (degrees)')
axes[0].set_title('PC1 Instability vs Eigenvalue Gap')
axes[0].grid(alpha=0.3)

# Visual: small gap (unstable) vs large gap (stable)
for ax_idx, (l1, l2, label) in enumerate([(2.1, 1.9, 'Small gap (λ₁−λ₂=0.2)'),
                                           (3.5, 0.5, 'Large gap (λ₁−λ₂=3.0)')]):
    Sigma = np.diag([l1, l2])
    L     = np.linalg.cholesky(Sigma)
    trial_pcs = []
    for _ in range(10):
        X = (L @ np.random.randn(2, n)).T
        S = X.T @ X / n
        _, vecs = np.linalg.eigh(S)
        trial_pcs.append(vecs[:, -1])

axes[1].set_aspect('equal')
colors_trial = ['red', 'orange', 'blue', 'purple', 'green',
                'brown', 'pink', 'gray', 'cyan', 'black']
for l1, l2, ls, label in [(2.1, 1.9, '--', 'Small gap'), (3.5, 0.5, '-', 'Large gap')]:
    Sigma = np.diag([l1, l2])
    L     = np.linalg.cholesky(Sigma)
    for trial_i in range(8):
        X = (L @ np.random.randn(2, n)).T
        S = X.T @ X / n
        _, vecs = np.linalg.eigh(S)
        u = vecs[:, -1]
        axes[1].annotate('', xy=0.8*u, xytext=-0.8*u,
                         arrowprops=dict(arrowstyle='->', color=colors_trial[trial_i],
                                         lw=1.5 if ls=='-' else 0.8,
                                         linestyle=ls))
    axes[1].plot([], [], color='black', ls=ls, lw=2, label=label)

axes[1].set_xlim(-1.1, 1.1); axes[1].set_ylim(-1.1, 1.1)
axes[1].set_title('PC1 Direction Across 8 Random Seeds')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('u₁'); axes[1].set_ylabel('u₂')

plt.suptitle('Eigenvalue Gap and PC Stability', fontsize=12)
plt.tight_layout()
plt.show()

print('\nSummary: when eigenvalues are close, PC direction is arbitrary')
print('→ downstream models trained on unstable PCs will behave inconsistently on new data')